In [59]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

geo = {
    "Chamonix" :{
        "latitude": 45.922761997253296,
        "longitude": 6.871686726139134
        },
    "Annecy": {
        "latitude": 45.899247,
        "longitude": 6.129384
        },
    "Saint-Gervais-les-Bains": {
        "latitude": 45.8897,
        "longitude": 6.7119
        }
}

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"

for city, coordinates in geo.items():
	params = {
		"latitude": coordinates["latitude"],
		"longitude": coordinates["longitude"],
		"start_date": "2022-12-31",
		"end_date": "2025-12-31",
		"hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"],
		"timezone": "GMT",
	}
	responses = openmeteo.weather_api(url, params = params)

	# Process first location. Add a for-loop for multiple locations or weather models
	response = responses[0]

	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
	hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
	hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
	hourly_wind_speed_10m = hourly.Variables(3).ValuesAsNumpy()

	hourly_data = {
		"date": pd.date_range(
			start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
			end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
			freq = pd.Timedelta(seconds = hourly.Interval()),
			inclusive = "left"
		).tz_convert("Europe/Paris")
	}

	hourly_data["Ville"] = city
	hourly_data["temperature_2m"] = hourly_temperature_2m
	hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
	hourly_data["precipitation"] = hourly_precipitation
	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

	hourly_dataframe = pd.DataFrame(data = hourly_data)